### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql_1 = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    oi.is_free_gift,
    ci.customer_hierarchy,
    ci.customer_type,
    ci.channel_source, 
    si.store_id,
    si.store_name,
    si.store_type,
    si.store_city,
    si.store_region,
    si.store_country,
    si.latitude,
    si.longitude
FROM "Order" o
LEFT JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
LEFT JOIN "CustomerInfo" ci
    ON o.customer_id = ci.customer_id
LEFT JOIN "StoreInfo" si
    ON ci.preferred_store_id = si.store_id
WHERE o.order_status IN ('Completed', 'Shipped')
"""

df_order_completed_shipped = pd.read_sql(sql_1, engine)

# 导出数据
df_order_completed_shipped.to_parquet('2023-2024_sale_perform_by_customer_store.parquet', engine='fastparquet', index=False)
print("数据已保存为 2023-2024_sale_perform_by_customer_store.parquet 文件")
# 查看数据
df_order_completed_shipped

数据已保存为 2023-2024_sale_by_customer_store.parquet 文件


,order_id,customer_id,order_date,order_status,total_price_before_tax,product_id,quantity,product_name,unit_price,line_price_before_tax,...,customer_hierarchy,customer_type,channel_source,store_name,store_type,store_city,store_region,store_country,latitude,longitude
0,55,32187,2023-02-11 12:42:12,Completed,180.10,123,1,Ray-Ban Mega Wayfarer Optics Prescription,180.10,180.10,...,3,VIP Loyal Customers,Offline,Ray-Ban House New York,Ray-Ban Store,House New York,New York,United States,42.638406,-73.755585
1,70,34359,2023-03-17 00:44:16,Completed,187.60,19,1,Ray-Ban Sam Non-prescription,177.60,177.60,...,4,Regular Customers,Offline,Ray-Ban Santa Clara,Ray-Ban Store,Santa Clara,California,United States,36.794826,-119.492491
2,79,28626,2023-02-03 21:00:09,Completed,160.64,35,1,Ray-Ban Original Wayfarer Non-prescription,160.64,160.64,...,3,VIP Loyal Customers,Offline,Ray-Ban San Diego,Ray-Ban Store,San Diego,California,United States,36.761941,-119.413001
3,137,3695,2023-03-21 20:07:14,Completed,145.43,153,1,Ray-Ban Clubmaster Prescription,177.35,145.43,...,3,VIP Loyal Customers,Offline,Ray-Ban Georgetown,Ray-Ban Store,Georgetown,District of Columbia,United States,39.779954,-98.577908
4,146,12062,2023-03-15 20:56:43,Completed,237.84,40,1,Ray-Ban Ray-Ban Meta Prescription,237.84,237.84,...,3,VIP Loyal Customers,Offline,Ray-Ban Santa Clara,Ray-Ban Store,Santa Clara,California,United States,36.794826,-119.492491
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655896,499963,72391,2024-11-25 16:09:59,Shipped,615.53,180,1,Essilor Mirror Progressive,233.88,152.02,...,0,Promotional Sensitive Customers,Offline,Ray-Ban Bellevue,Ray-Ban Store,Bellevue,Washington,United States,47.754979,-120.744099
655897,499963,72391,2024-11-25 16:09:59,Shipped,615.53,42,1,Ray-Ban Ray-Ban Reverse Non-prescription,411.78,267.66,...,0,Promotional Sensitive Customers,Offline,Ray-Ban Bellevue,Ray-Ban Store,Bellevue,Washington,United States,47.754979,-120.744099
655898,499972,77650,2024-11-20 17:51:07,Completed,559.22,134,1,Ray-Ban Ray-Ban Reverse Prescription,321.38,321.38,...,4,Regular Customers,Online,Ray-Ban La Jolla,Ray-Ban Store,La Jolla,California,United States,36.775713,-119.427092
655899,499972,77650,2024-11-20 17:51:07,Completed,559.22,40,1,Ray-Ban Ray-Ban Meta Prescription,237.84,237.84,...,4,Regular Customers,Online,Ray-Ban La Jolla,Ray-Ban Store,La Jolla,California,United States,36.775713,-119.427092


### agg_sales_by_store —— 门店销售聚合

In [ ]:
# 查询数据
sql_2 = """
SELECT
    si.store_id,
    si.store_name,
    si.store_type,
    si.store_city,
    si.store_region,
    si.store_country,
    si.latitude,
    si.longitude,
    
    EXTRACT(YEAR FROM o.order_date)::INT AS year,
    EXTRACT(MONTH FROM o.order_date)::INT AS month,
    TO_CHAR(o.order_date, 'YYYY-MM') AS year_month,

    SUM(
        CASE
            WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
            THEN oi.line_price_before_tax
            ELSE 0
        END
    ) AS total_sales_amount,

    SUM(
        CASE
            WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
            THEN oi.quantity
            ELSE 0
        END
    ) AS total_sales_qty,

    SUM(
        CASE
            WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
            THEN oi.line_price_before_tax
                 - (COALESCE(pi.cost_price, 0) * oi.quantity)
            ELSE 0
        END
    ) AS gross_profit,

    CASE
        WHEN SUM(
            CASE
                WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
                THEN oi.line_price_before_tax
                ELSE 0
            END
        ) = 0
        THEN NULL

        ELSE ROUND(
            SUM(
                CASE
                    WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
                    THEN oi.line_price_before_tax
                         - (COALESCE(pi.cost_price, 0) * oi.quantity)
                    ELSE 0
                END
            )::NUMERIC
            /
            SUM(
                CASE
                    WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
                    THEN oi.line_price_before_tax
                    ELSE 0
                END
            )::NUMERIC,
            4
        )
    END AS gross_margin_rate,

    COUNT(DISTINCT o.order_id) AS order_count,

    COUNT(DISTINCT o.customer_id) AS customer_count,

    CASE
        WHEN COUNT(DISTINCT o.order_id) = 0
        THEN NULL

        ELSE ROUND(
            SUM(
                CASE
                    WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
                    THEN oi.line_price_before_tax
                    ELSE 0
                END
            )::NUMERIC
            /
            COUNT(DISTINCT o.order_id)::NUMERIC,
            2
        )
    END AS aov

FROM "Order" o

LEFT JOIN "OrderItem" oi
    ON o.order_id = oi.order_id

LEFT JOIN "CustomerInfo" ci
    ON o.customer_id = ci.customer_id

LEFT JOIN "StoreInfo" si
    ON ci.preferred_store_id = si.store_id

LEFT JOIN "ProductInfo" pi
    ON oi.product_id = pi.product_id

WHERE o.order_status IN ('Completed', 'Shipped')

GROUP BY
    si.store_id,
    si.store_name,
    si.store_type,
    si.store_city,
    si.store_region,
    si.store_country,
    year,
    month,
    year_month

ORDER BY
    si.store_id,
    year,
    month
"""

df_order_completed_shipped = pd.read_sql(sql_2, engine)

# 导出数据
df_order_completed_shipped.to_parquet('2023-2024_sale_perform_by_customer_store.parquet', engine='fastparquet', index=False)
print("数据已保存为 2023-2024_sale_perform_by_customer_store.parquet 文件")
# 查看数据
df_order_completed_shipped

数据已保存为 2023-2024_sale_perform_by_customer_store.parquet 文件


,store_id,store_name,store_type,store_city,store_region,store_country,latitude,longitude,year,month,year_month,total_sales_amount,total_sales_qty,gross_profit,gross_margin_rate,order_count,customer_count,aov
0,1,Ray-Ban La Jolla,Ray-Ban Store,La Jolla,California,United States,36.775713,-119.427092,2023,1,2023-01,149038.79,643,80808.26,0.5422,397,378,375.41
1,1,Ray-Ban La Jolla,Ray-Ban Store,La Jolla,California,United States,36.775713,-119.427092,2023,2,2023-02,124596.23,510,67410.85,0.5410,317,297,393.05
2,1,Ray-Ban La Jolla,Ray-Ban Store,La Jolla,California,United States,36.775713,-119.427092,2023,3,2023-03,120426.68,508,65065.57,0.5403,316,299,381.10
3,1,Ray-Ban La Jolla,Ray-Ban Store,La Jolla,California,United States,36.775713,-119.427092,2023,4,2023-04,152767.89,623,82365.87,0.5392,367,346,416.26
4,1,Ray-Ban La Jolla,Ray-Ban Store,La Jolla,California,United States,36.775713,-119.427092,2023,5,2023-05,148373.94,614,80631.58,0.5434,372,355,398.85
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2419,101,LensCrafters Macy's - Oakbrook,Multi-brand Retail,Oak Brook,Illinois,United States,40.609009,-89.390987,2024,8,2024-08,30789.88,122,16902.49,0.5490,67,67,459.55
2420,101,LensCrafters Macy's - Oakbrook,Multi-brand Retail,Oak Brook,Illinois,United States,40.609009,-89.390987,2024,9,2024-09,22400.55,87,12479.86,0.5571,56,54,400.01
2421,101,LensCrafters Macy's - Oakbrook,Multi-brand Retail,Oak Brook,Illinois,United States,40.609009,-89.390987,2024,10,2024-10,44735.42,183,23184.15,0.5183,108,97,414.22
2422,101,LensCrafters Macy's - Oakbrook,Multi-brand Retail,Oak Brook,Illinois,United States,40.609009,-89.390987,2024,11,2024-11,42118.55,175,21914.02,0.5203,107,100,393.63


### agg_sales_by_product —— 产品销售聚合

In [9]:
# 查询数据
sql_3 = """
SELECT
    pi.product_id,
    pi.category,
    pi.sub_category,
    pi.brand,

    EXTRACT(YEAR FROM o.order_date)::INT AS year,
    EXTRACT(MONTH FROM o.order_date)::INT AS month,
    TO_CHAR(o.order_date, 'YYYY-MM') AS year_month,

    SUM(
        CASE
            WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
            THEN oi.line_price_before_tax
            ELSE 0
        END
    ) AS total_sales_amount,

    SUM(
        CASE
            WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
            THEN oi.quantity
            ELSE 0
        END
    ) AS total_sales_qty,

    SUM(
        CASE
            WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
            THEN oi.line_price_before_tax
                 - (COALESCE(pi.cost_price, 0) * oi.quantity)
            ELSE 0
        END
    ) AS gross_profit,

    CASE
        WHEN SUM(
            CASE
                WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
                THEN oi.line_price_before_tax
                ELSE 0
            END
        ) = 0
        THEN NULL

        ELSE ROUND(
            SUM(
                CASE
                    WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
                    THEN oi.line_price_before_tax
                         - (COALESCE(pi.cost_price, 0) * oi.quantity)
                    ELSE 0
                END
            )::NUMERIC
            /
            SUM(
                CASE
                    WHEN oi.is_free_gift = FALSE OR oi.is_free_gift IS NULL
                    THEN oi.line_price_before_tax
                    ELSE 0
                END
            )::NUMERIC,
            4
        )
    END AS gross_margin_rate,

    COUNT(DISTINCT o.order_id) AS order_count

FROM "Order" o

LEFT JOIN "OrderItem" oi
    ON o.order_id = oi.order_id

LEFT JOIN "PromotionActivity" pa
    ON o.campaign_id = pa.campaign_id

LEFT JOIN "ProductInfo" pi
    ON oi.product_id = pi.product_id

LEFT JOIN "CustomerInfo" ci
    ON o.customer_id = ci.customer_id

LEFT JOIN "StoreInfo" si
    ON ci.preferred_store_id = si.store_id

WHERE o.order_status IN ('Completed', 'Shipped')

GROUP BY
    pi.product_id,
    pi.category,
    pi.sub_category,
    pi.brand,
    year,
    month,
    year_month

ORDER BY
    pi.product_id,
    year,
    month
"""
df_order_completed_shipped = pd.read_sql(sql_3, engine)

# 导出数据
df_order_completed_shipped.to_parquet('2023-2024_sale_perform_by_customer_product.parquet', engine='fastparquet', index=False)
print("数据已保存为 2023-2024_sale_perform_by_customer_product.parquet 文件")
# 查看数据
df_order_completed_shipped

数据已保存为 2023-2024_sale_perform_by_customer_product.parquet 文件


,product_id,category,sub_category,brand,year,month,year_month,total_sales_amount,total_sales_qty,gross_profit,gross_margin_rate,order_count
0,1,Sunglasses,Classic Sunglasses,Ray-Ban,2023,1,2023-01,26645.64,167,13347.43,0.5009,159
1,1,Sunglasses,Classic Sunglasses,Ray-Ban,2023,2,2023-02,24221.60,151,12197.47,0.5036,141
2,1,Sunglasses,Classic Sunglasses,Ray-Ban,2023,3,2023-03,24665.30,156,12243.02,0.4964,132
3,1,Sunglasses,Classic Sunglasses,Ray-Ban,2023,4,2023-04,29929.99,186,15118.81,0.5051,161
4,1,Sunglasses,Classic Sunglasses,Ray-Ban,2023,5,2023-05,32424.96,202,16339.70,0.5039,181
...,...,...,...,...,...,...,...,...,...,...,...,...
4975,220,Accessory,Free Gift,Ray-Ban,2024,5,2024-05,0.00,0,0.00,NaN,30
4976,220,Accessory,Free Gift,Ray-Ban,2024,6,2024-06,0.00,0,0.00,NaN,30
4977,220,Accessory,Free Gift,Ray-Ban,2024,10,2024-10,0.00,0,0.00,NaN,39
4978,220,Accessory,Free Gift,Ray-Ban,2024,11,2024-11,0.00,0,0.00,NaN,35


### Data validate

### cal product type

In [20]:
df_product_categories = df_order_completed_shipped['category'].unique()
df_product_categories

array(['Eyeglasses', 'Lens', 'Sunglasses', 'AI Glasses', 'Accessory'],
      dtype=object)

In [21]:
df_product_sub_categories = df_order_completed_shipped['sub_category'].unique()
df_product_sub_categories

array(['Eyeglasses', 'Violet Lens', 'Classic Sunglasses',
       'Transitions Lens', 'Evolve Lens', 'AI Smart Glasses',
       'Mirror Lens', 'Polarized+ Lens', 'Gradient Lens',
       'Solid Color Lens', 'Chromance Lens', 'Polarized S Lens',
       'Clear Lens', 'Free Gift'], dtype=object)

In [22]:
df_product_brands = df_order_completed_shipped['brand'].unique()
df_product_brands

array(['Ray-Ban', 'Essilor'], dtype=object)

In [23]:
df_customer_types = df_order_completed_shipped['customer_type'].unique()
df_customer_types

array(['Promotional Sensitive Customers', 'Regular Customers',
       'Lens Customers', 'VIP Loyal Customers', 'One-time Customers'],
      dtype=object)

In [26]:
df_store_types = df_order_completed_shipped['store_type'].unique()
df_store_types

array(['Ray-Ban Store', 'Multi-brand Retail'], dtype=object)

In [27]:
df_store_regions = df_order_completed_shipped['store_region'].unique()
df_store_regions

array(['Florida', 'California', 'Illinois', 'New York', 'Ohio',
       'Massachusetts', 'Georgia', 'New Jersey', 'Texas',
       'District of Columbia', 'Colorado', 'Hawaii', 'Washington',
       'Nevada', 'Tennessee', 'North Carolina'], dtype=object)